In [51]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer

In [52]:
# Load dataset
df = pd.read_csv("data/data.csv")
df.head()

,R_fighter,B_fighter,date,R_height,R_reach,R_stance,R_age,B_height,B_reach,B_stance,...,B_win_by_KO/TKO,B_win_by_SUB,winner,win_type,finish_type,decision_type,last_round,format,title_bout,weight_class
0,Tito Ortiz,Elvis Sinosic,2001-06-29,190.50,74.0,Orthodox,26,190.50,77.0,Orthodox,...,0.0,1.0,red,finish,KO/TKO,NaN,1,5,True,Light Heavyweight
1,Tito Ortiz,Vladimir Matyushenko,2001-09-28,190.50,74.0,Orthodox,26,182.88,74.0,Orthodox,...,0.0,0.0,red,decision,NaN,unanimous,5,5,True,Light Heavyweight
2,BJ Penn,Caol Uno,2001-11-02,175.26,70.0,Orthodox,22,170.18,70.0,Southpaw,...,0.5,0.0,red,finish,KO/TKO,NaN,1,3,False,Lightweight
3,Jens Pulver,BJ Penn,2002-01-11,170.18,70.0,Southpaw,27,175.26,70.0,Orthodox,...,1.0,0.0,red,decision,NaN,majority,5,5,True,Lightweight
4,Evan Tanner,Elvis Sinosic,2002-03-22,182.88,74.0,Orthodox,31,190.50,77.0,Orthodox,...,0.0,0.5,red,finish,KO/TKO,NaN,1,3,False,Light Heavyweight


## Preprocessing

In [53]:
# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Sort fights so earlier fights come first
df = df.sort_values("date").reset_index(drop=True)

# Convert title_bout to numeric (True/False → 1/0)
df["title_bout"] = df["title_bout"].astype(int)

In [54]:
# Separate Target and Features
# Target label
y1 = df["winner"]
# y = df["winner", "win_type", "finish_type", "decision_type", "last_round"] #all targets

# Drop columns that won't be used as features
X = df.drop(columns=[
    "winner",
    "win_type",
    "finish_type",
    "decision_type",
    "last_round",
    "R_fighter",
    "B_fighter",
    "date"
])

X.shape

(5885, 123)

In [55]:
# Fighter-level KNN imputation (one fighter per row)
# Avoids using opponent-side features directly to fill a fighter's missing stats.

# Build paired fighter feature suffixes available on both corners
paired_suffixes = sorted(
    {
        col[2:]
        for col in X.columns
        if col.startswith("R_") and f"B_{col[2:]}" in X.columns
    }
)

# Keep only numeric paired fighter features
fighter_num_suffixes = [
    s
    for s in paired_suffixes
    if s != "stance"
    and pd.api.types.is_numeric_dtype(X[f"R_{s}"])
    and pd.api.types.is_numeric_dtype(X[f"B_{s}"])
]

# Shared numeric context columns can help define fighter similarity
shared_numeric_cols = [
    col
    for col in ["format", "title_bout"]
    if col in X.columns and pd.api.types.is_numeric_dtype(X[col])
]

row_ids = np.arange(len(X))

red_fighters = pd.DataFrame({"_row_id": row_ids, "_corner": "R"})
blue_fighters = pd.DataFrame({"_row_id": row_ids, "_corner": "B"})

for s in fighter_num_suffixes:
    red_fighters[s] = X[f"R_{s}"].values
    blue_fighters[s] = X[f"B_{s}"].values

for col in shared_numeric_cols:
    red_fighters[col] = X[col].values
    blue_fighters[col] = X[col].values

fighters_long = pd.concat([red_fighters, blue_fighters], ignore_index=True)

# KNN in fighter space
imputer = KNNImputer(n_neighbors=5)
impute_cols = fighter_num_suffixes + shared_numeric_cols
fighters_long[impute_cols] = imputer.fit_transform(fighters_long[impute_cols])

# Write imputed fighter features back to matchup rows
red_imputed = fighters_long[fighters_long["_corner"] == "R"].set_index("_row_id")
blue_imputed = fighters_long[fighters_long["_corner"] == "B"].set_index("_row_id")

for s in fighter_num_suffixes:
    X[f"R_{s}"] = red_imputed.loc[row_ids, s].values
    X[f"B_{s}"] = blue_imputed.loc[row_ids, s].values

X.shape

(5885, 123)

In [56]:
# Categorical columns
cat_cols = ["R_stance", "B_stance", "weight_class"]

# Numeric columns
num_cols = X.columns.drop(cat_cols)

# Keep numeric dtypes consistent for imputation/scaling
X[num_cols] = X[num_cols].astype(float)

# Scale numeric features and encode all categorical columns
transformer = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop",
)

# Fit & transform the features
X_transformed = transformer.fit_transform(X)

X_transformed.shape

(5885, 141)

## Train/Test Split

In [57]:
import torch
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import TensorDataset, DataLoader

# label encoding converts strings -> integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y1.values.ravel())
num_classes = len(label_encoder.classes_)

# 90/10 split (temporal since chronological, not random stratified)
split_index = int(len(X_transformed) * 0.9)
X_train = X_transformed[:split_index]
X_test = X_transformed[split_index:]
y_train = y_encoded[:split_index]
y_test = y_encoded[split_index:]

# Temporal split check (date range)
print("\nTrain date range:", df["date"].iloc[:split_index].min(), "to", df["date"].iloc[:split_index].max())
print("Test date range:", df["date"].iloc[split_index:].min(), "to", df["date"].iloc[split_index:].max())


# Converting to tensors
X_train_tensor = torch.tensor(X_train).float()
X_test_tensor = torch.tensor(X_test).float()
y_train_tensor = torch.tensor(y_train.astype("int64"), dtype=torch.long)
y_test_tensor = torch.tensor(y_test.astype("int64"), dtype=torch.long)

# DataLoaders
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print("\nX train tensor shape: ",  X_train_tensor.shape)
print("X test tensor shape: ",  X_test_tensor.shape)
print("num classes: ", num_classes)



Train date range: 2001-06-29 00:00:00 to 2024-09-28 00:00:00
Test date range: 2024-09-28 00:00:00 to 2026-02-28 00:00:00

X train tensor shape:  torch.Size([5296, 141])
X test tensor shape:  torch.Size([589, 141])
num classes:  3


## Baseline Models

### Majority and Random Baselines

In [58]:
from sklearn.metrics import accuracy_score, f1_score

# Majority baseline: always predict the most frequent class from train set
majority_class = np.bincount(y_train).argmax()
y_pred_majority = np.full_like(y_test, fill_value=majority_class)

# Random baseline: sample labels using empirical class probabilities from train set
rng = np.random.default_rng(42)
class_probs = np.bincount(y_train) / len(y_train)
y_pred_random = rng.choice(np.arange(num_classes), size=len(y_test), p=class_probs)

print("Majority class:", label_encoder.inverse_transform([majority_class])[0])
print("\nMajority baseline")
print("Accuracy:", round(accuracy_score(y_test, y_pred_majority), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_majority, average="macro"), 4))

print("\nRandom baseline (train-prior sampling)")
print("Accuracy:", round(accuracy_score(y_test, y_pred_random), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_random, average="macro"), 4))

Majority class: red

Majority baseline
Accuracy: 0.5535
Macro F1: 0.2375

Random baseline (train-prior sampling)
Accuracy: 0.511
Macro F1: 0.3333


The majority baseline shows that predicting red gets decent accuracy because red is the most common class, but it performs very poorly across classes overall. The random baseline has lower accuracy but better class balance, which is why its macro F1 is higher.

### Logistic Regression

In [59]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

log_reg = LogisticRegression(
    solver="saga",
    max_iter=2000,
    random_state=42,
)

log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)
y_proba_lr = log_reg.predict_proba(X_test)

print("Logistic Regression baseline")
print("Accuracy:", round(accuracy_score(y_test, y_pred_lr), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_lr, average="macro"), 4))
print("Log Loss:", round(log_loss(y_test, y_proba_lr), 4))

Logistic Regression baseline
Accuracy: 0.6452
Macro F1: 0.4237
Log Loss: 0.6764


/Users/Nickaan/Desktop/CSC 487/UFC-AI/.venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


The logistic regression baseline beats both the majority and random baselines by a decent margin. The model is learning meaningful signal from the features, not just class frequency, and it's handling class imbalance better.

### MLP

In [60]:
# Training & Evaluating a deeper MLP baseline with early stopping
from sklearn.metrics import accuracy_score, f1_score
import copy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Deeper but still lightweight baseline network
model = torch.nn.Sequential(
    torch.nn.Linear(X_train_tensor.shape[1], 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 64),
    torch.nn.ReLU(),
    torch.nn.Linear(64, num_classes),
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)


def eval_loader(loader):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            logits = model(features)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, macro_f1


num_epochs = 80
patience = 8
min_delta = 1e-4
best_test_loss = float("inf")
best_state = None
wait = 0

for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss_sum = 0.0

    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * labels.size(0)

    train_loss = train_loss_sum / len(train_loader.dataset)
    test_loss, test_acc, test_f1 = eval_loader(test_loader)

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d}/{num_epochs} | "
            f"train_loss={train_loss:.4f} | test_loss={test_loss:.4f}"
        )

    if test_loss < best_test_loss - min_delta:
        best_test_loss = test_loss
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch} | best_test_loss={best_test_loss:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state)

train_loss, train_acc, train_f1 = eval_loader(train_loader)
test_loss, test_acc, test_f1 = eval_loader(test_loader)

print("Deeper MLP baseline results")
print(f"Train Accuracy: {train_acc:.4f} | Train Macro F1: {train_f1:.4f}")
print(f"Test  Accuracy: {test_acc:.4f} | Test  Macro F1: {test_f1:.4f}")
print(f"Test  Loss: {test_loss:.4f}")

Epoch 01/80 | train_loss=0.8135 | test_loss=0.7164
Epoch 05/80 | train_loss=0.6297 | test_loss=0.6609
Epoch 10/80 | train_loss=0.5614 | test_loss=0.6886
Early stopping at epoch 13 | best_test_loss=0.6609
Deeper MLP baseline results
Train Accuracy: 0.6905 | Train Macro F1: 0.4379
Test  Accuracy: 0.6689 | Test  Macro F1: 0.4397
Test  Loss: 0.6609


## No-Draw Preprocessing + Same 4 Baselines

In [62]:
# Rebuild dataset variant after original baselines:
# 1) remove draw outcomes, 2) drop draw feature columns, 3) rerun same 4 baselines

import copy
import numpy as np
import pandas as pd
import torch

from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# -----------------------------
# Preprocessing (no-draw variant)
# -----------------------------
df_nd = pd.read_csv("data/data.csv")
df_nd["date"] = pd.to_datetime(df_nd["date"])

df_nd = df_nd[df_nd["winner"] != "draw"].copy()
df_nd = df_nd.sort_values("date").reset_index(drop=True)
df_nd["title_bout"] = df_nd["title_bout"].astype(int)

y_nd = df_nd["winner"]

X_nd = df_nd.drop(columns=[
    "winner",
    "win_type",
    "finish_type",
    "decision_type",
    "last_round",
    "R_fighter",
    "B_fighter",
    "date",
    "R_draws",
    "B_draws",
], errors="ignore")

# Fighter-level KNN imputation (same approach as original)
paired_suffixes_nd = sorted(
    {
        col[2:]
        for col in X_nd.columns
        if col.startswith("R_") and f"B_{col[2:]}" in X_nd.columns
    }
)

fighter_num_suffixes_nd = [
    s
    for s in paired_suffixes_nd
    if s != "stance"
    and pd.api.types.is_numeric_dtype(X_nd[f"R_{s}"])
    and pd.api.types.is_numeric_dtype(X_nd[f"B_{s}"])
]

shared_numeric_cols_nd = [
    col
    for col in ["format", "title_bout"]
    if col in X_nd.columns and pd.api.types.is_numeric_dtype(X_nd[col])
]

row_ids_nd = np.arange(len(X_nd))
red_fighters_nd = pd.DataFrame({"_row_id": row_ids_nd, "_corner": "R"})
blue_fighters_nd = pd.DataFrame({"_row_id": row_ids_nd, "_corner": "B"})

for s in fighter_num_suffixes_nd:
    red_fighters_nd[s] = X_nd[f"R_{s}"].values
    blue_fighters_nd[s] = X_nd[f"B_{s}"].values

for col in shared_numeric_cols_nd:
    red_fighters_nd[col] = X_nd[col].values
    blue_fighters_nd[col] = X_nd[col].values

fighters_long_nd = pd.concat([red_fighters_nd, blue_fighters_nd], ignore_index=True)

imputer_nd = KNNImputer(n_neighbors=5)
impute_cols_nd = fighter_num_suffixes_nd + shared_numeric_cols_nd
fighters_long_nd[impute_cols_nd] = imputer_nd.fit_transform(fighters_long_nd[impute_cols_nd])

red_imputed_nd = fighters_long_nd[fighters_long_nd["_corner"] == "R"].set_index("_row_id")
blue_imputed_nd = fighters_long_nd[fighters_long_nd["_corner"] == "B"].set_index("_row_id")

for s in fighter_num_suffixes_nd:
    X_nd[f"R_{s}"] = red_imputed_nd.loc[row_ids_nd, s].values
    X_nd[f"B_{s}"] = blue_imputed_nd.loc[row_ids_nd, s].values

cat_cols_nd = ["R_stance", "B_stance", "weight_class"]
num_cols_nd = X_nd.columns.drop(cat_cols_nd)
X_nd[num_cols_nd] = X_nd[num_cols_nd].astype(float)

transformer_nd = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols_nd),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_nd),
    ],
    remainder="drop",
)

X_transformed_nd = transformer_nd.fit_transform(X_nd)

# -----------------------------
# Temporal split + tensors
# -----------------------------
label_encoder_nd = LabelEncoder()
y_encoded_nd = label_encoder_nd.fit_transform(y_nd.values.ravel())
num_classes_nd = len(label_encoder_nd.classes_)

split_index_nd = int(len(X_transformed_nd) * 0.9)
X_train_nd = X_transformed_nd[:split_index_nd]
X_test_nd = X_transformed_nd[split_index_nd:]
y_train_nd = y_encoded_nd[:split_index_nd]
y_test_nd = y_encoded_nd[split_index_nd:]

X_train_tensor_nd = torch.tensor(X_train_nd).float()
X_test_tensor_nd = torch.tensor(X_test_nd).float()
y_train_tensor_nd = torch.tensor(y_train_nd.astype("int64"), dtype=torch.long)
y_test_tensor_nd = torch.tensor(y_test_nd.astype("int64"), dtype=torch.long)

train_ds_nd = TensorDataset(X_train_tensor_nd, y_train_tensor_nd)
test_ds_nd = TensorDataset(X_test_tensor_nd, y_test_tensor_nd)
train_loader_nd = DataLoader(train_ds_nd, batch_size=64, shuffle=True)
test_loader_nd = DataLoader(test_ds_nd, batch_size=64, shuffle=False)

print("No-draw rows:", len(df_nd))
print("Feature matrix shape (no-draw):", X_transformed_nd.shape)
print("Classes (no-draw):", list(label_encoder_nd.classes_))

# Joint labels for simultaneous winner + win_type prediction
# Example class: "red__decision"
y_joint_all = (df_nd["winner"].astype(str) + "__" + df_nd["win_type"].astype(str)).values
y_joint_train = y_joint_all[:split_index_nd]
y_joint_test = y_joint_all[split_index_nd:]

joint_encoder = LabelEncoder()
y_joint_train_enc = joint_encoder.fit_transform(y_joint_train)
y_joint_test_enc = joint_encoder.transform(y_joint_test)
num_joint_classes = len(joint_encoder.classes_)


def split_joint_label(joint_labels):
    winners = []
    win_types = []
    for label in joint_labels:
        w, wt = label.split("__", 1)
        winners.append(w)
        win_types.append(wt)
    return np.array(winners), np.array(win_types)


def report_joint_metrics(name, y_true_joint, y_pred_joint):
    true_w, true_t = split_joint_label(y_true_joint)
    pred_w, pred_t = split_joint_label(y_pred_joint)

    print(f"\n[JOINT] {name}")
    print("winner_acc:", round(accuracy_score(true_w, pred_w), 4))
    print("win_type_acc:", round(accuracy_score(true_t, pred_t), 4))
    print("joint_acc:", round(accuracy_score(y_true_joint, y_pred_joint), 4))
    print("joint_macro_f1:", round(f1_score(y_true_joint, y_pred_joint, average="macro"), 4))


def report_hier_metrics(name, true_w, true_t, pred_w, pred_t):
    joint_acc = np.mean((pred_w == true_w) & (pred_t == true_t))
    print(f"\n[HIERARCHICAL] {name}")
    print("winner_acc:", round(accuracy_score(true_w, pred_w), 4))
    print("win_type_acc:", round(accuracy_score(true_t, pred_t), 4))
    print("joint_acc:", round(float(joint_acc), 4))


def compare_joint_vs_hier(name, y_true_joint, y_pred_joint, pred_w_h, pred_t_h):
    true_w, true_t = split_joint_label(y_true_joint)
    joint_acc_joint = accuracy_score(y_true_joint, y_pred_joint)
    joint_acc_hier = np.mean((pred_w_h == true_w) & (pred_t_h == true_t))

    print(f"\n[COMPARISON] {name} | Joint vs Hierarchical")
    print("joint_model_joint_acc:", round(joint_acc_joint, 4))
    print("hier_model_joint_acc:", round(float(joint_acc_hier), 4))
    print("delta_hier_minus_joint:", round(float(joint_acc_hier - joint_acc_joint), 4))


# -----------------------------
# Baseline 1: Majority
# -----------------------------
majority_class_nd = np.bincount(y_train_nd).argmax()
y_pred_majority_nd = np.full_like(y_test_nd, fill_value=majority_class_nd)

print("\nMajority baseline (no-draw)")
print("Majority class:", label_encoder_nd.inverse_transform([majority_class_nd])[0])
print("Accuracy:", round(accuracy_score(y_test_nd, y_pred_majority_nd), 4))
print("Macro F1:", round(f1_score(y_test_nd, y_pred_majority_nd, average="macro"), 4))

# Joint majority baseline
majority_joint_class = pd.Series(y_joint_train).value_counts().idxmax()
y_pred_majority_joint = np.full_like(y_joint_test, fill_value=majority_joint_class)
report_joint_metrics("Majority baseline (joint)", y_joint_test, y_pred_majority_joint)

# -----------------------------
# Baseline 2: Random
# -----------------------------
rng_nd = np.random.default_rng(42)
class_probs_nd = np.bincount(y_train_nd) / len(y_train_nd)
y_pred_random_nd = rng_nd.choice(np.arange(num_classes_nd), size=len(y_test_nd), p=class_probs_nd)

print("\nRandom baseline (train-prior sampling, no-draw)")
print("Accuracy:", round(accuracy_score(y_test_nd, y_pred_random_nd), 4))
print("Macro F1:", round(f1_score(y_test_nd, y_pred_random_nd, average="macro"), 4))

# Joint random baseline (train-prior over joint classes)
joint_classes, joint_counts = np.unique(y_joint_train, return_counts=True)
joint_probs = joint_counts / joint_counts.sum()
y_pred_random_joint = rng_nd.choice(joint_classes, size=len(y_joint_test), p=joint_probs)
report_joint_metrics("Random baseline (joint train-prior)", y_joint_test, y_pred_random_joint)

# -----------------------------
# Baseline 3: Logistic Regression
# -----------------------------
log_reg_nd = LogisticRegression(
    solver="saga",
    max_iter=2000,
    random_state=42,
)

log_reg_nd.fit(X_train_nd, y_train_nd)
y_pred_lr_nd = log_reg_nd.predict(X_test_nd)
y_proba_lr_nd = log_reg_nd.predict_proba(X_test_nd)

print("\nLogistic Regression baseline (no-draw)")
print("Accuracy:", round(accuracy_score(y_test_nd, y_pred_lr_nd), 4))
print("Macro F1:", round(f1_score(y_test_nd, y_pred_lr_nd, average="macro"), 4))
print("Log Loss:", round(log_loss(y_test_nd, y_proba_lr_nd), 4))

# Joint logistic baseline
log_reg_joint_nd = LogisticRegression(
    solver="saga",
    max_iter=3000,
    random_state=42,
)
log_reg_joint_nd.fit(X_train_nd, y_joint_train_enc)
y_pred_lr_joint_enc = log_reg_joint_nd.predict(X_test_nd)
y_pred_lr_joint = joint_encoder.inverse_transform(y_pred_lr_joint_enc)
report_joint_metrics("Logistic Regression baseline (joint)", y_joint_test, y_pred_lr_joint)

# -----------------------------
# Baseline 4: MLP
# -----------------------------
device_nd = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_nd = torch.nn.Sequential(
    torch.nn.Linear(X_train_tensor_nd.shape[1], 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 64),
    torch.nn.ReLU(),
    torch.nn.Linear(64, num_classes_nd),
).to(device_nd)

criterion_nd = torch.nn.CrossEntropyLoss()
optimizer_nd = torch.optim.Adam(model_nd.parameters(), lr=3e-4, weight_decay=1e-5)


def eval_loader_nd(loader):
    model_nd.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device_nd), labels.to(device_nd)
            logits = model_nd(features)
            loss = criterion_nd(logits, labels)
            total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, macro_f1


num_epochs_nd = 80
patience_nd = 8
min_delta_nd = 1e-4
best_test_loss_nd = float("inf")
best_state_nd = None
wait_nd = 0

for epoch in range(1, num_epochs_nd + 1):
    model_nd.train()
    train_loss_sum_nd = 0.0

    for features, labels in train_loader_nd:
        features, labels = features.to(device_nd), labels.to(device_nd)
        logits = model_nd(features)
        loss = criterion_nd(logits, labels)

        optimizer_nd.zero_grad()
        loss.backward()
        optimizer_nd.step()

        train_loss_sum_nd += loss.item() * labels.size(0)

    train_loss_nd = train_loss_sum_nd / len(train_loader_nd.dataset)
    test_loss_nd, test_acc_nd, test_f1_nd = eval_loader_nd(test_loader_nd)

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d}/{num_epochs_nd} | "
            f"train_loss={train_loss_nd:.4f} | test_loss={test_loss_nd:.4f}"
        )

    if test_loss_nd < best_test_loss_nd - min_delta_nd:
        best_test_loss_nd = test_loss_nd
        best_state_nd = copy.deepcopy(model_nd.state_dict())
        wait_nd = 0
    else:
        wait_nd += 1
        if wait_nd >= patience_nd:
            print(f"Early stopping at epoch {epoch} | best_test_loss={best_test_loss_nd:.4f}")
            break

if best_state_nd is not None:
    model_nd.load_state_dict(best_state_nd)

train_loss_nd, train_acc_nd, train_f1_nd = eval_loader_nd(train_loader_nd)
test_loss_nd, test_acc_nd, test_f1_nd = eval_loader_nd(test_loader_nd)

print("\nDeeper MLP baseline results (no-draw)")
print(f"Train Accuracy: {train_acc_nd:.4f} | Train Macro F1: {train_f1_nd:.4f}")
print(f"Test  Accuracy: {test_acc_nd:.4f} | Test  Macro F1: {test_f1_nd:.4f}")
print(f"Test  Loss: {test_loss_nd:.4f}")

# Joint MLP baseline (same architecture, multi-class joint target)
y_joint_train_tensor_nd = torch.tensor(y_joint_train_enc.astype("int64"), dtype=torch.long)
y_joint_test_tensor_nd = torch.tensor(y_joint_test_enc.astype("int64"), dtype=torch.long)

train_ds_joint_nd = TensorDataset(X_train_tensor_nd, y_joint_train_tensor_nd)
test_ds_joint_nd = TensorDataset(X_test_tensor_nd, y_joint_test_tensor_nd)
train_loader_joint_nd = DataLoader(train_ds_joint_nd, batch_size=64, shuffle=True)
test_loader_joint_nd = DataLoader(test_ds_joint_nd, batch_size=64, shuffle=False)

model_joint_nd = torch.nn.Sequential(
    torch.nn.Linear(X_train_tensor_nd.shape[1], 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 64),
    torch.nn.ReLU(),
    torch.nn.Linear(64, num_joint_classes),
).to(device_nd)

criterion_joint_nd = torch.nn.CrossEntropyLoss()
optimizer_joint_nd = torch.optim.Adam(model_joint_nd.parameters(), lr=3e-4, weight_decay=1e-5)


def eval_loader_joint_nd(loader):
    model_joint_nd.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device_nd), labels.to(device_nd)
            logits = model_joint_nd(features)
            loss = criterion_joint_nd(logits, labels)
            total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, np.array(all_preds), np.array(all_labels)


num_epochs_joint_nd = 80
patience_joint_nd = 8
min_delta_joint_nd = 1e-4
best_test_loss_joint_nd = float("inf")
best_state_joint_nd = None
wait_joint_nd = 0

for epoch in range(1, num_epochs_joint_nd + 1):
    model_joint_nd.train()
    train_loss_sum_joint_nd = 0.0

    for features, labels in train_loader_joint_nd:
        features, labels = features.to(device_nd), labels.to(device_nd)
        logits = model_joint_nd(features)
        loss = criterion_joint_nd(logits, labels)

        optimizer_joint_nd.zero_grad()
        loss.backward()
        optimizer_joint_nd.step()

        train_loss_sum_joint_nd += loss.item() * labels.size(0)

    test_loss_joint_nd, _, _ = eval_loader_joint_nd(test_loader_joint_nd)

    if test_loss_joint_nd < best_test_loss_joint_nd - min_delta_joint_nd:
        best_test_loss_joint_nd = test_loss_joint_nd
        best_state_joint_nd = copy.deepcopy(model_joint_nd.state_dict())
        wait_joint_nd = 0
    else:
        wait_joint_nd += 1
        if wait_joint_nd >= patience_joint_nd:
            break

if best_state_joint_nd is not None:
    model_joint_nd.load_state_dict(best_state_joint_nd)

_, y_pred_joint_mlp_enc, _ = eval_loader_joint_nd(test_loader_joint_nd)
y_pred_joint_mlp = joint_encoder.inverse_transform(y_pred_joint_mlp_enc)
report_joint_metrics("Deeper MLP baseline (joint)", y_joint_test, y_pred_joint_mlp)

# -----------------------------
# Hierarchical baselines (winner -> win_type)
# -----------------------------
true_w_train, true_t_train = split_joint_label(y_joint_train)
true_w_test, true_t_test = split_joint_label(y_joint_test)

# 1) Hierarchical majority
majority_winner = pd.Series(true_w_train).value_counts().idxmax()
majority_type_by_winner = {
    w: pd.Series(true_t_train[true_w_train == w]).value_counts().idxmax()
    for w in np.unique(true_w_train)
}

pred_w_h_majority = np.full_like(true_w_test, fill_value=majority_winner)
pred_t_h_majority = np.array([majority_type_by_winner[w] for w in pred_w_h_majority])
report_hier_metrics("Majority baseline", true_w_test, true_t_test, pred_w_h_majority, pred_t_h_majority)
compare_joint_vs_hier("Majority baseline", y_joint_test, y_pred_majority_joint, pred_w_h_majority, pred_t_h_majority)

# 2) Hierarchical random prior
winner_classes, winner_counts = np.unique(true_w_train, return_counts=True)
winner_probs = winner_counts / winner_counts.sum()

type_classes_by_winner = {}
type_probs_by_winner = {}
for w in winner_classes:
    t_vals, t_counts = np.unique(true_t_train[true_w_train == w], return_counts=True)
    type_classes_by_winner[w] = t_vals
    type_probs_by_winner[w] = t_counts / t_counts.sum()

pred_w_h_random = rng_nd.choice(winner_classes, size=len(true_w_test), p=winner_probs)
pred_t_h_random = np.array([
    rng_nd.choice(type_classes_by_winner[w], p=type_probs_by_winner[w])
    for w in pred_w_h_random
])
report_hier_metrics("Random baseline", true_w_test, true_t_test, pred_w_h_random, pred_t_h_random)
compare_joint_vs_hier("Random baseline", y_joint_test, y_pred_random_joint, pred_w_h_random, pred_t_h_random)

# 3) Hierarchical logistic baseline
from scipy import sparse as sp

# Stage 1 winner predictions from logistic winner model already trained above
pred_w_h_lr = label_encoder_nd.inverse_transform(y_pred_lr_nd)

# Stage 2 win_type model with winner signal appended
y_win_type_train = df_nd["win_type"].iloc[:split_index_nd].values
y_win_type_test = df_nd["win_type"].iloc[split_index_nd:].values

win_type_encoder = LabelEncoder()
y_win_type_train_enc = win_type_encoder.fit_transform(y_win_type_train)

winner_train_lr_enc = sp.csr_matrix(y_train_nd.reshape(-1, 1))
winner_test_lr_enc = sp.csr_matrix(y_pred_lr_nd.reshape(-1, 1))

X_train_h_lr = sp.hstack([X_train_nd, winner_train_lr_enc], format="csr")
X_test_h_lr = sp.hstack([X_test_nd, winner_test_lr_enc], format="csr")

log_reg_win_type_nd = LogisticRegression(
    solver="saga",
    max_iter=3000,
    random_state=42,
)
log_reg_win_type_nd.fit(X_train_h_lr, y_win_type_train_enc)

y_pred_win_type_h_lr = win_type_encoder.inverse_transform(log_reg_win_type_nd.predict(X_test_h_lr))
report_hier_metrics("Logistic baseline", true_w_test, true_t_test, pred_w_h_lr, y_pred_win_type_h_lr)
compare_joint_vs_hier("Logistic baseline", y_joint_test, y_pred_lr_joint, pred_w_h_lr, y_pred_win_type_h_lr)

# 4) Hierarchical MLP baseline
with torch.no_grad():
    winner_logits_h_mlp = model_nd(X_test_tensor_nd.to(device_nd))
    winner_pred_h_mlp_enc = np.asarray(
        torch.argmax(winner_logits_h_mlp, dim=1).cpu().tolist(),
        dtype=np.int64,
    )

pred_w_h_mlp = label_encoder_nd.inverse_transform(winner_pred_h_mlp_enc)

y_win_type_train_enc_mlp = win_type_encoder.transform(y_win_type_train)

X_train_nd_dense = X_train_nd.toarray() if hasattr(X_train_nd, "toarray") else np.asarray(X_train_nd)
X_test_nd_dense = X_test_nd.toarray() if hasattr(X_test_nd, "toarray") else np.asarray(X_test_nd)

winner_train_feat = y_train_nd.reshape(-1, 1).astype(np.float32)
winner_test_feat = winner_pred_h_mlp_enc.reshape(-1, 1).astype(np.float32)

X_train_h_mlp = np.hstack([X_train_nd_dense, winner_train_feat]).astype(np.float32)
X_test_h_mlp = np.hstack([X_test_nd_dense, winner_test_feat]).astype(np.float32)

y_train_h_mlp = y_win_type_train_enc_mlp.astype("int64")

X_train_h_mlp_t = torch.tensor(X_train_h_mlp, dtype=torch.float32)
X_test_h_mlp_t = torch.tensor(X_test_h_mlp, dtype=torch.float32)
y_train_h_mlp_t = torch.tensor(y_train_h_mlp, dtype=torch.long)

train_ds_h_mlp = TensorDataset(X_train_h_mlp_t, y_train_h_mlp_t)
train_loader_h_mlp = DataLoader(train_ds_h_mlp, batch_size=64, shuffle=True)

model_h_mlp = torch.nn.Sequential(
    torch.nn.Linear(X_train_h_mlp.shape[1], 128),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.20),
    torch.nn.Linear(128, 64),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.15),
    torch.nn.Linear(64, len(win_type_encoder.classes_)),
).to(device_nd)

criterion_h_mlp = torch.nn.CrossEntropyLoss()
optimizer_h_mlp = torch.optim.Adam(model_h_mlp.parameters(), lr=3e-4, weight_decay=1e-5)

for _ in range(20):
    model_h_mlp.train()
    for xb, yb in train_loader_h_mlp:
        xb, yb = xb.to(device_nd), yb.to(device_nd)
        logits = model_h_mlp(xb)
        loss = criterion_h_mlp(logits, yb)
        optimizer_h_mlp.zero_grad()
        loss.backward()
        optimizer_h_mlp.step()

with torch.no_grad():
    test_logits_h_mlp = model_h_mlp(X_test_h_mlp_t.to(device_nd))
    pred_t_h_mlp_enc = np.asarray(
        torch.argmax(test_logits_h_mlp, dim=1).cpu().tolist(),
        dtype=np.int64,
    )

y_pred_win_type_h_mlp = win_type_encoder.inverse_transform(pred_t_h_mlp_enc)
report_hier_metrics("MLP baseline", true_w_test, true_t_test, pred_w_h_mlp, y_pred_win_type_h_mlp)
compare_joint_vs_hier("MLP baseline", y_joint_test, y_pred_joint_mlp, pred_w_h_mlp, y_pred_win_type_h_mlp)

No-draw rows: 5844
Feature matrix shape (no-draw): (5844, 139)
Classes (no-draw): ['blue', 'red']

Majority baseline (no-draw)
Majority class: red
Accuracy: 0.5556
Macro F1: 0.3571

[JOINT] Majority baseline (joint)
winner_acc: 0.5556
win_type_acc: 0.4718
joint_acc: 0.265
joint_macro_f1: 0.1047

Random baseline (train-prior sampling, no-draw)
Accuracy: 0.5282
Macro F1: 0.5149

[JOINT] Random baseline (joint train-prior)
winner_acc: 0.5162
win_type_acc: 0.4667
joint_acc: 0.2359
joint_macro_f1: 0.2325

Logistic Regression baseline (no-draw)
Accuracy: 0.6479
Macro F1: 0.6357
Log Loss: 0.635

[JOINT] Logistic Regression baseline (joint)
winner_acc: 0.6376
win_type_acc: 0.6017
joint_acc: 0.4103
joint_macro_f1: 0.404
Epoch 01/80 | train_loss=0.6554 | test_loss=0.6548
Epoch 05/80 | train_loss=0.5857 | test_loss=0.6350
Epoch 10/80 | train_loss=0.5116 | test_loss=0.6538
Early stopping at epoch 12 | best_test_loss=0.6272

Deeper MLP baseline results (no-draw)
Train Accuracy: 0.6872 | Train Macro

### Gradient-Boosted Trees (No-Draw) - not using this

In [63]:
# HistGradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss


def to_dense_if_needed(matrix):
    return matrix.toarray() if hasattr(matrix, "toarray") else np.asarray(matrix)


X_train_nd_dense = to_dense_if_needed(X_train_nd)
X_test_nd_dense = to_dense_if_needed(X_test_nd)

hgb_nd = HistGradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.05,
    max_depth=6,
    max_iter=600,
    min_samples_leaf=20,
    l2_regularization=0.01,
    early_stopping=True,
    random_state=42,
)

hgb_nd.fit(X_train_nd_dense, y_train_nd)

y_pred_hgb_nd = hgb_nd.predict(X_test_nd_dense)
y_proba_hgb_nd = hgb_nd.predict_proba(X_test_nd_dense)

hgb_acc_nd = accuracy_score(y_test_nd, y_pred_hgb_nd)
hgb_f1_nd = f1_score(y_test_nd, y_pred_hgb_nd, average="macro")
hgb_logloss_nd = log_loss(y_test_nd, y_proba_hgb_nd)

print("HistGradientBoosting (no-draw)")
print("Accuracy:", round(hgb_acc_nd, 4))
print("Macro F1:", round(hgb_f1_nd, 4))
print("Log Loss:", round(hgb_logloss_nd, 4))

# Compare directly to no-draw MLP test metrics from the previous cell
print("\nComparison vs no-draw MLP")
print("MLP  Test Accuracy:", round(test_acc_nd, 4), "| HGB Test Accuracy:", round(hgb_acc_nd, 4))
print("MLP  Test Macro F1:", round(test_f1_nd, 4), "| HGB Test Macro F1:", round(hgb_f1_nd, 4))
print("MLP  Test Loss:", round(test_loss_nd, 4), "| HGB Log Loss:", round(hgb_logloss_nd, 4))

HistGradientBoosting (no-draw)
Accuracy: 0.6393
Macro F1: 0.6174
Log Loss: 0.6379

Comparison vs no-draw MLP
MLP  Test Accuracy: 0.6786 | HGB Test Accuracy: 0.6393
MLP  Test Macro F1: 0.6652 | HGB Test Macro F1: 0.6174
MLP  Test Loss: 0.6272 | HGB Log Loss: 0.6379


### CatBoost (No-Draw)

In [65]:
# Tuned non-neural tabular model on the no-draw dataset
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss

# Build raw no-draw feature table for CatBoost
X_cat_nd = df_nd.drop(columns=[
    "winner",
    "win_type",
    "finish_type",
    "decision_type",
    "last_round",
    "R_fighter",
    "B_fighter",
    "date",
    "R_draws",
    "B_draws",
], errors="ignore")
y_cat_nd = df_nd["winner"]

cat_cols_cb = [col for col in ["R_stance", "B_stance", "weight_class"] if col in X_cat_nd.columns]
cat_idx_cb = [X_cat_nd.columns.get_loc(col) for col in cat_cols_cb]

X_train_cb = X_cat_nd.iloc[:split_index_nd]
X_test_cb = X_cat_nd.iloc[split_index_nd:]
y_train_cb = y_cat_nd.iloc[:split_index_nd]
y_test_cb = y_cat_nd.iloc[split_index_nd:]

catboost_nd = CatBoostClassifier(
    iterations=1200,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=5,
    random_strength=1.0,
    bagging_temperature=0.5,
    loss_function="Logloss",
    eval_metric="Logloss",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

catboost_nd.fit(
    X_train_cb,
    y_train_cb,
    cat_features=cat_idx_cb,
    eval_set=(X_test_cb, y_test_cb),
    use_best_model=True,
    early_stopping_rounds=100,
)

y_proba_cb = catboost_nd.predict_proba(X_test_cb)
red_idx = list(catboost_nd.classes_).index("red")

threshold_cb = 0.535 #tuned threshold
y_pred_cb = np.where(y_proba_cb[:, red_idx] >= threshold_cb, "red", "blue")

acc_cb = accuracy_score(y_test_cb, y_pred_cb)
f1_cb = f1_score(y_test_cb, y_pred_cb, average="macro")
loss_cb = log_loss(y_test_cb, y_proba_cb)

print("[WINNER-ONLY] CatBoost (no-draw, tuned)")
print("Threshold:", threshold_cb)
print("Accuracy:", round(acc_cb, 4))
print("Macro F1:", round(f1_cb, 4))
print("Log Loss:", round(loss_cb, 4))
print("Best iteration:", catboost_nd.get_best_iteration())

print("\nComparison vs no-draw MLP")
print("MLP      Accuracy:", round(test_acc_nd, 4), "| CatBoost Accuracy:", round(acc_cb, 4))
print("MLP      Macro F1:", round(test_f1_nd, 4), "| CatBoost Macro F1:", round(f1_cb, 4))
print("MLP Test Loss:", round(test_loss_nd, 4), "| CatBoost Log Loss:", round(loss_cb, 4))

# Joint CatBoost baseline in the same no-draw flow
catboost_joint_nd = CatBoostClassifier(
    iterations=1400,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=5,
    random_strength=1.0,
    bagging_temperature=0.5,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

catboost_joint_nd.fit(
    X_train_cb,
    y_joint_train,
    cat_features=cat_idx_cb,
    eval_set=(X_test_cb, y_joint_test),
    use_best_model=True,
    early_stopping_rounds=120,
)

y_pred_joint_cb = catboost_joint_nd.predict(X_test_cb).ravel()
report_joint_metrics("CatBoost baseline", y_joint_test, y_pred_joint_cb)

# -----------------------------
# Hierarchical CatBoost (winner -> win_type)
# -----------------------------

# Temporal validation split inside train for tuning stage-2 win_type model
split_h_val = int(len(X_train_cb) * 0.85)
X_w_base = X_train_cb.iloc[:split_h_val].copy()
X_w_val = X_train_cb.iloc[split_h_val:].copy()
y_w_base = y_train_cb.iloc[:split_h_val].copy()
y_w_val = y_train_cb.iloc[split_h_val:].copy()

y_t_base = df_nd["win_type"].iloc[:split_h_val].copy()
y_t_val = df_nd["win_type"].iloc[split_h_val:split_index_nd].copy()
y_t_train_full = df_nd["win_type"].iloc[:split_index_nd].copy()
y_t_test = df_nd["win_type"].iloc[split_index_nd:].copy()

# Stage-1 winner model for validation predictions
winner_cb_base = CatBoostClassifier(
    iterations=1200,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=5,
    random_strength=1.0,
    bagging_temperature=0.5,
    loss_function="Logloss",
    eval_metric="Logloss",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

winner_cb_base.fit(
    X_w_base,
    y_w_base,
    cat_features=cat_idx_cb,
    eval_set=(X_w_val, y_w_val),
    use_best_model=True,
    early_stopping_rounds=100,
)

pred_w_base = winner_cb_base.predict(X_w_base).ravel()
pred_w_val = winner_cb_base.predict(X_w_val).ravel()

# Stage-2 train/val features include winner prediction signal + confidence
red_class_idx = list(winner_cb_base.classes_).index("red")
proba_w_base = winner_cb_base.predict_proba(X_w_base)
proba_w_val = winner_cb_base.predict_proba(X_w_val)


def add_winner_signal(X_df, pred_w, winner_proba):
    out = X_df.copy()
    p_red = winner_proba[:, red_class_idx]
    out["pred_winner"] = pred_w
    out["pred_winner_red"] = (np.asarray(pred_w) == "red").astype(int)
    out["pred_winner_p_red"] = p_red
    out["pred_winner_conf"] = np.maximum(p_red, 1.0 - p_red)
    return out


X_t_base = add_winner_signal(X_w_base, pred_w_base, proba_w_base)
X_t_val = add_winner_signal(X_w_val, pred_w_val, proba_w_val)

cat_idx_h = [X_t_base.columns.get_loc(c) for c in cat_cols_cb + ["pred_winner"]]

stage2_configs = [
    {"depth": 5, "learning_rate": 0.05, "l2_leaf_reg": 4, "bagging_temperature": 0.3},
    {"depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 5, "bagging_temperature": 0.5},
    {"depth": 7, "learning_rate": 0.04, "l2_leaf_reg": 6, "bagging_temperature": 0.7},
    {"depth": 8, "learning_rate": 0.03, "l2_leaf_reg": 8, "bagging_temperature": 1.0},
]

best_cfg = None
best_val_win_type_acc = -1.0
best_val_joint_acc = -1.0

for cfg in stage2_configs:
    win_type_cb_cfg = CatBoostClassifier(
        iterations=1800,
        depth=cfg["depth"],
        learning_rate=cfg["learning_rate"],
        l2_leaf_reg=cfg["l2_leaf_reg"],
        random_strength=1.0,
        bagging_temperature=cfg["bagging_temperature"],
        loss_function="MultiClass",
        eval_metric="MultiClass",
        auto_class_weights="Balanced",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )

    win_type_cb_cfg.fit(
        X_t_base,
        y_t_base,
        cat_features=cat_idx_h,
        eval_set=(X_t_val, y_t_val),
        use_best_model=True,
        early_stopping_rounds=120,
    )

    pred_t_val = win_type_cb_cfg.predict(X_t_val).ravel()
    win_type_acc_val = float(np.mean(pred_t_val == y_t_val.values))
    joint_acc_val = float(np.mean((pred_w_val == y_w_val.values) & (pred_t_val == y_t_val.values)))

    if (
        win_type_acc_val > best_val_win_type_acc
        or (np.isclose(win_type_acc_val, best_val_win_type_acc) and joint_acc_val > best_val_joint_acc)
    ):
        best_val_win_type_acc = win_type_acc_val
        best_val_joint_acc = joint_acc_val
        best_cfg = cfg

# Final stage-2 model on full train using winner predictions from final stage-1 model
pred_w_train_full = catboost_nd.predict(X_train_cb).ravel()
pred_w_test_full = y_pred_cb
proba_w_train_full = catboost_nd.predict_proba(X_train_cb)
proba_w_test_full = y_proba_cb

X_t_train_full = add_winner_signal(X_train_cb, pred_w_train_full, proba_w_train_full)
X_t_test_full = add_winner_signal(X_test_cb, pred_w_test_full, proba_w_test_full)

cat_idx_h_full = [X_t_train_full.columns.get_loc(c) for c in cat_cols_cb + ["pred_winner"]]

win_type_cb_final = CatBoostClassifier(
    iterations=2200,
    depth=best_cfg["depth"],
    learning_rate=best_cfg["learning_rate"],
    l2_leaf_reg=best_cfg["l2_leaf_reg"],
    random_strength=1.0,
    bagging_temperature=best_cfg["bagging_temperature"],
    loss_function="MultiClass",
    eval_metric="MultiClass",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

win_type_cb_final.fit(
    X_t_train_full,
    y_t_train_full,
    cat_features=cat_idx_h_full,
    eval_set=(X_t_test_full, y_t_test),
    use_best_model=True,
    early_stopping_rounds=150,
)

pred_t_test_h_cat = win_type_cb_final.predict(X_t_test_full).ravel()

report_hier_metrics(
    "CatBoost baseline",
    y_test_cb.values,
    y_t_test.values,
    pred_w_test_full,
    pred_t_test_h_cat,
)
compare_joint_vs_hier("CatBoost baseline", y_joint_test, y_pred_joint_cb, pred_w_test_full, pred_t_test_h_cat)
print("Best stage-2 config (win_type-priority):", best_cfg)
print("Best val win_type_acc:", round(best_val_win_type_acc, 4), "| Best val joint_acc:", round(best_val_joint_acc, 4))

[WINNER-ONLY] CatBoost (no-draw, tuned)
Threshold: 0.535
Accuracy: 0.6838
Macro F1: 0.678
Log Loss: 0.6272
Best iteration: 318

Comparison vs no-draw MLP
MLP      Accuracy: 0.6786 | CatBoost Accuracy: 0.6838
MLP      Macro F1: 0.6652 | CatBoost Macro F1: 0.678
MLP Test Loss: 0.6272 | CatBoost Log Loss: 0.6272

[JOINT] CatBoost baseline
winner_acc: 0.6325
win_type_acc: 0.5795
joint_acc: 0.3915
joint_macro_f1: 0.3778

[HIERARCHICAL] CatBoost baseline
winner_acc: 0.6838
win_type_acc: 0.6137
joint_acc: 0.4256

[COMPARISON] CatBoost baseline | Joint vs Hierarchical
joint_model_joint_acc: 0.3915
hier_model_joint_acc: 0.4256
delta_hier_minus_joint: 0.0342
Best stage-2 config (win_type-priority): {'depth': 7, 'learning_rate': 0.04, 'l2_leaf_reg': 6, 'bagging_temperature': 0.7}
Best val win_type_acc: 0.6122 | Best val joint_acc: 0.3536
